In [2]:
# Jaxlib
import jax
from jax import lax
from jax import random as jrnd
from jax import numpy as jnp
from jax import tree_util as jtu
jax.config.update('jax_enable_x64', True)

# Others
from matplotlib import pyplot as plt

# This
from numerics import *
from seismic import *

In [3]:
Mw = 6.3
site = Site(0., 0., 760., 0.5, 2.9, 1.)
erf1 = ERF(9., 5., 5., 8.)
erf2 = ERF(9., 4., 4., 8.)
erf3 = ERF(10., 5., 4., 5.)
fault1 = Fault(0., 20., 3.0, 0.1, 40., 80., 90., 2.5, 0., erf1)
fault2 = Fault(0., 25., 0.5, 0.2, 80., 80., 30., 0.5, 1., erf2)
fault3 = Fault(22., 12.1, 3.9, 1.2, 130., 40., 22., 3., 1., erf3)
fault_tree = make_fault_tree(fault1, fault2, fault3)
scn = Scenario(site, fault_tree)

In [4]:
# xyz Distance from rupture surface
def calc_R_rup(site: Site, fault: Fault) -> float:
    """Calculate 3d distance between site and nearest part of rupture"""
    # Grab coordinates
    site_xyz = site.calc_xyz()
    fault_xyz_hyp = fault.calc_xyz_hyp()
    fault_dxyz_tor = fault.calc_dxyz_tor()
    # Overall distance between hypocenter and TOR
    fault_dr_tor = jnp.linalg.norm(fault_dxyz_tor, ord = 2)
    # Distance for down-dip edge (edge2)
    scaling = (fault.width - fault_dr_tor) / fault_dr_tor
    # Fault edges
    edge1_xyz = fault_xyz_hyp + fault_dxyz_tor
    edge2_xyz = fault_xyz_hyp - fault_dxyz_tor * scaling

    # Distance vectors
    edge1_dxyz = edge1_xyz - site_xyz
    edge2_dxyz = edge2_xyz - site_xyz
    span_xyz = edge2_xyz - edge1_xyz

    # Distances
    edge1_r = jnp.linalg.norm(edge1_dxyz, ord = 2)
    edge2_r = jnp.linalg.norm(edge2_dxyz, ord = 2)
    span_xyz_cross = jnp.linalg.cross(span_xyz, edge1_dxyz)
    span_r = jnp.linalg.norm(span_xyz_cross, ord = 2) / jnp.linalg.norm(span_xyz, ord = 2)
    proj_xyz = edge1_xyz + (edge1_dxyz @ span_xyz) / (span_xyz @ span_xyz) * span_xyz
    proj_z = proj_xyz[-1]

    # Condition for edge vs. other distance
    edge_r = jnp.minimum(edge1_r, edge2_r)
    edge_cond = jnp.logical_or(proj_z > edge2_xyz[-1], proj_z < fault.z_tor)
    return lax.select(edge_cond, edge_r, span_r)

# xy Distance from epicenter
def calc_R_epi(site: Site, fault: Fault) -> float:
    """Calculate 2d distance between site and hypocenter."""
    return jnp.linalg.norm(site.calc_xy() - fault.calc_xy_hyp(), ord=2)

# xyz Distance from hypocenter
def calc_R_hyp(site: Site, fault: Fault) -> float:
    """Calculate 3d distance between site and hypocenter."""
    return jnp.linalg.norm(site.calc_xyz() - fault.calc_xyz_hyp(), ord=2)

# xy Distance from top of rupture
def calc_R_x(site: Site, fault: Fault) -> float:
    """Calculate 2d distance between site and top of rupture."""
    fault_xyz_tor = fault.calc_xyz_hyp() + fault.calc_dxyz_tor()
    return jnp.linalg.norm(site.calc_xyz() - fault_xyz_tor, ord=2)

def calc_R(site:Site, fault:Fault):
    """Calculate distances of interest."""
    return [calc_R_jb(site, fault), calc_R_rup(site, fault), calc_R_epi(site, fault), calc_R_hyp(site, fault), calc_R_x(site, fault)]

In [5]:
f_ASK14(Mw, 0.5, site, fault1, calc_R(site, fault1))

(Array(-1.66017022, dtype=float64), Array(0.64235816, dtype=float64))

In [8]:
from jax.scipy.stats.norm import cdf as gaussian_cdf
from time import time

@jtu.register_pytree_node_class
class GMMLT:
    def __init__(self, T:float, gmms:list, weights:jax.typing.ArrayLike):
        self.T = T
        self.gmms = gmms
        self.weights = weights

    # Calculate for a single GMM. Takes R so we don't repeat the calculation every time.
    def calc_single(self, i:int, Mw:float, site:Site, fault:Fault, R:jax.Array):
        return lax.switch(i, self.gmms, Mw, self.T, site, fault, R)

    def tree_flatten(self):
        return (self.weights), self.gmms
    
    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(aux, *children)

def calc_haz(x:float, M_min:float, gmms:GMMLT, scn:Scenario, dM:float = 0.1):
    t0 = time()
    fault_num = scn.fault_tree.x.shape[0]
    M_min, M_max = scn.fault_tree.erf.M_min, scn.fault_tree.erf.M_max

    # Midpoint quadrature over maximum range
    roots_M = jnp.arange(M_min.min() + dM / 2, M_max.max() + dM / 2, dM)
    # Array of ones/zeros for each fault signifying array inside/outside range (shape (roots_M.shape, fault_num))
    weights_mask = (roots_M[:, None] > M_min[None, :]) & (roots_M[:, None] < M_max[None, :])
    # Quadrature weights
    weights_M = ((M_max - M_min) / weights_mask.sum(axis = 0))
    weights_M = jnp.einsum('ij,j->ij', weights_mask, weights_M)

    # ERF incremental rates
    n_M = jax.vmap(scn.fault_tree.erf.calc_n)(roots_M)

    # Ground motion means + stds for each fault at Legendre roots
    # Calculate R so we don't need to do it every time
    R_tree = jax.vmap(calc_R, in_axes = (None, 0))(scn.site, scn.fault_tree)
    # Triple vmap. First, across faults (and corresponding distances)
    calc_faults = jax.vmap(gmms.calc_single, in_axes=(None, None, None, 0, 0))
    # Then across magnitudes
    calc_M = jax.vmap(calc_faults, in_axes=(None, 0, None, None, None))
    # Then across GMMs. This order minimizes recompilation
    calc_gmms = jax.vmap(calc_M, in_axes=(0, None, None, None, None))
    # Grab indices and vmap across
    gmm_idcs = jnp.arange(len(gmms.gmms))
    all_mu_lnSA, all_std_lnSA = calc_gmms(gmm_idcs, roots_M, scn.site, scn.fault_tree, R_tree)
    # And weight according to logic tree
    mu_lnSA = jnp.einsum('i,ijk->jk', gmms.weights, all_mu_lnSA)
    std_lnSA = jnp.einsum('i,ijk->jk', gmms.weights, all_std_lnSA)
    
    # Probabilities of exceedance 
    prob_x = 1 - gaussian_cdf(jnp.log(x), mu_lnSA, std_lnSA)
    # Hazard integrand (magnitude probabilities * exceedance probabilities)
    haz_intgrnd = prob_x * n_M
    # Integrate using mag weights
    haz = jnp.einsum('ij,ij->', weights_M, haz_intgrnd)
    return haz

x = 0.3
Mw_min = 4.
gmm_backbones = [f_ASK14, f_BSSA14, f_CB14, f_Idriss14, f_CY14]
gmm_epi_plus = []#[f_epistemic_AAY14(gmm, 3.18) for gmm in gmm_backbones]
gmm_epi_minus = []#[f_epistemic_AAY14(gmm, -3.18) for gmm in gmm_backbones]
gmms = gmm_backbones + gmm_epi_plus + gmm_epi_minus
weights = jnp.ones(len(gmms)) / len(gmms)
gmmlt = GMMLT(0.9, gmms, weights)
R = calc_R(site, fault1)
#print(gmmlt.calc_single(1, 5., site, fault1, R))

for x in [0.001, 0.01, 0.1]:
    haz = calc_haz(x, Mw_min, gmmlt, scn)
    print(haz)

# with jax.profiler.trace("/tmp/jax-trace"):
#     for gmm,gmm_name in zip(gmm_backbones, ['ASK','BSSA','CB','Idriss','CY']):
#         with jax.named_scope(gmm_name):
#             gm = gmm(Mw, 0.5, site, fault1, R)
#             jax.block_until_ready(gm)

5.177518013629513e-08
4.461028282034562e-08
3.7572468103619483e-08
